In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
# download pre-trained model
pretrained_weights = ViT_B_16_Weights.IMAGENET1K_V1
model = vit_b_16(weights=pretrained_weights).to(device)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 192MB/s]


In [4]:
# patch the model to probe attention maps and intermediates
def layer_forward(self, input: torch.Tensor):
    torch._assert(input.dim() == 3, f"Expected (batch_size, seq_length, hidden_dim) got {input.shape}")
    x = self.ln_1(input)
    normalized_intermediates.append(x.detach().cpu()) # probe
    x, att_map = self.self_attention(x, x, x, need_weights=True, average_attn_weights=False)
    att_maps.append(att_map.detach().cpu()) # probe
    x = self.dropout(x)
    x = x + input
    y = self.ln_2(x)
    y = self.mlp(y)
    o = x + y
    intermediates.append(o.detach().cpu()) # probe
    return o

for layer in model.encoder.layers:
    layer.forward = layer_forward.__get__(layer, type(layer))

In [5]:
import kagglehub
path = kagglehub.dataset_download("ambityga/imagenet100")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imagenet100' dataset.
Path to dataset files: /kaggle/input/imagenet100


In [6]:
# download ImageNet Labels
!wget http://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt

--2026-01-21 11:26:57--  http://storage.googleapis.com/bit_models/ilsvrc2012_wordnet_lemmas.txt
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.145.207, 74.125.128.207, 74.125.143.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.145.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21675 (21K) [text/plain]
Saving to: ‘ilsvrc2012_wordnet_lemmas.txt’

ilsvrc2012_wordnet_ 100%[===================>]  21.17K  --.-KB/s    in 0s      

2026-01-21 11:26:57 (249 MB/s) - ‘ilsvrc2012_wordnet_lemmas.txt’ saved [21675/21675]



In [7]:
imagenet_labels = dict(enumerate(open('ilsvrc2012_wordnet_lemmas.txt')))
print(imagenet_labels)

{0: 'tench, Tinca_tinca\n', 1: 'goldfish, Carassius_auratus\n', 2: 'great_white_shark, white_shark, man-eater, man-eating_shark, Carcharodon_carcharias\n', 3: 'tiger_shark, Galeocerdo_cuvieri\n', 4: 'hammerhead, hammerhead_shark\n', 5: 'electric_ray, crampfish, numbfish, torpedo\n', 6: 'stingray\n', 7: 'cock\n', 8: 'hen\n', 9: 'ostrich, Struthio_camelus\n', 10: 'brambling, Fringilla_montifringilla\n', 11: 'goldfinch, Carduelis_carduelis\n', 12: 'house_finch, linnet, Carpodacus_mexicanus\n', 13: 'junco, snowbird\n', 14: 'indigo_bunting, indigo_finch, indigo_bird, Passerina_cyanea\n', 15: 'robin, American_robin, Turdus_migratorius\n', 16: 'bulbul\n', 17: 'jay\n', 18: 'magpie\n', 19: 'chickadee\n', 20: 'water_ouzel, dipper\n', 21: 'kite\n', 22: 'bald_eagle, American_eagle, Haliaeetus_leucocephalus\n', 23: 'vulture\n', 24: 'great_grey_owl, great_gray_owl, Strix_nebulosa\n', 25: 'European_fire_salamander, Salamandra_salamandra\n', 26: 'common_newt, Triturus_vulgaris\n', 27: 'eft\n', 28: 'sp

In [8]:
!ls -l $path/val.X/n01773549

total 7924
-rw-r--r-- 1 1000 1000   62175 Jan 20 23:04 ILSVRC2012_val_00002688.JPEG
-rw-r--r-- 1 1000 1000   97064 Jan 20 23:04 ILSVRC2012_val_00003431.JPEG
-rw-r--r-- 1 1000 1000   99968 Jan 20 23:04 ILSVRC2012_val_00004271.JPEG
-rw-r--r-- 1 1000 1000   90311 Jan 20 23:04 ILSVRC2012_val_00004335.JPEG
-rw-r--r-- 1 1000 1000   95669 Jan 20 23:04 ILSVRC2012_val_00007224.JPEG
-rw-r--r-- 1 1000 1000   70655 Jan 20 23:04 ILSVRC2012_val_00007882.JPEG
-rw-r--r-- 1 1000 1000   69880 Jan 20 23:04 ILSVRC2012_val_00008316.JPEG
-rw-r--r-- 1 1000 1000   97696 Jan 20 23:04 ILSVRC2012_val_00008352.JPEG
-rw-r--r-- 1 1000 1000  156532 Jan 20 23:04 ILSVRC2012_val_00009216.JPEG
-rw-r--r-- 1 1000 1000   70982 Jan 20 23:04 ILSVRC2012_val_00009558.JPEG
-rw-r--r-- 1 1000 1000   74675 Jan 20 23:04 ILSVRC2012_val_00011638.JPEG
-rw-r--r-- 1 1000 1000  306539 Jan 20 23:04 ILSVRC2012_val_00011677.JPEG
-rw-r--r-- 1 1000 1000   89467 Jan 20 23:04 ILSVRC2012_val_00011808.JPEG
-rw-r--r-- 1 1000 1000  258562 Jan 20 23

In [9]:
imagepath = path + '/val.X/' + 'n01773549' +'/' + 'ILSVRC2012_val_00008316.JPEG'

In [10]:
from google.colab import files
files.download(imagepath)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
image = cv2.imread(imagepath)
from google.colab.patches import cv2_imshow
cv2_imshow(image)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

In [ ]:
size = (224, 224)

In [ ]:
blob = transform(cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2RGB),size)).unsqueeze(0).to(device)

In [ ]:
blob.shape

In [ ]:
with torch.no_grad():
    att_maps = []
    intermediates = []
    normalized_intermediates = []
    logits = model(blob)

In [ ]:
# Present probabilities of categories
probabilities = torch.sigmoid(logits)[0]
labels = torch.argsort(probabilities, dim=-1, descending=True)
top5 = labels[:5]
print("Prediction:")
for idx in top5:
    print(f'{probs[idx.item()]:.5f} : {imagenet_labels[idx.item()]}', end='')

In [ ]:
att_maps = torch.cat(att_maps, dim=0)
intermediates = torch.cat(intermediates, dim=0)
normalized_intermediates = torch.cat(normalized_intermediates, dim=0)

In [ ]:
print(att_maps.shape)
print(intermediates.shape)
print(normalized_intermediates.shape)

In [ ]:
def draw_mask(img, mask):
    H, W = img.shape[:2]
    mask_resized = cv2.resize(mask, (W, H), interpolation=cv2.INTER_LINEAR)
    if img.ndim == 3:
        mask_resized = np.repeat(mask_resized[:, :, None], 3, axis=2)
    result = (img.astype(np.float32) * mask_resized).clip(0, 255).astype(np.uint8)
    return result

In [ ]:
base = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
num_layers, num_heads, N, _ = att_maps.shape
patch_size = int(math.sqrt(size[0]*size[1]//(N-1)))
att_size = (size[0]//patch_size, size[1]//patch_size)
plt.figure(figsize=(2 * num_heads, 2 * num_layers))
for t in range(num_layers):
    for i in range(num_heads):
        head = att_maps[t, i]
        mask = head[0, 1:].reshape(att_size[0], att_size[1])
        mask = mask.detach().cpu().numpy()
        disp = draw_mask(base, mask)
        plt.subplot(num_layers, num_heads, t * num_heads + i + 1)
        plt.imshow(disp, cmap='gray')
        plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# navrhnite vlastny sposob ako na obrazku najst (ako-tak) oblast s podobnym vyznamom ako ma bod udany promptom

In [ ]:
prompt = [200,350] # x, y # pavuk

In [ ]:
prompt = [200, 50] # x, y # zelena plocha